In [57]:
# Install libraries required for the RAG project

!pip install -q pypdf langchain langchain-community chromadb sentence-transformers
!pip install -q langchain-text-splitters

In [58]:
# Import the libraries required for document loading and text processing

import os
import langchain
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb

In [59]:
# Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [60]:
# Verify access to the ADA guideline documents and identify the PDF files

data_path = "/content/drive/MyDrive/ADA_Diabetes_Guidelines"

pdf_files = sorted(
    [file for file in os.listdir(data_path) if file.endswith(".pdf")]
)

print("PDF files found:")
for pdf in pdf_files:
    print(pdf)

PDF files found:
dc26s002.pdf
dc26s004.pdf
dc26s005.pdf
dc26s006.pdf
dc26s009.pdf
dc26s010.pdf


In [61]:
# Load the ADA guideline PDF documents and check the number of pages

for pdf in pdf_files:
    pdf_path = os.path.join(data_path, pdf)
    reader = PdfReader(pdf_path)
    print(f"{pdf}: {len(reader.pages)} pages")

dc26s002.pdf: 23 pages
dc26s004.pdf: 28 pages
dc26s005.pdf: 43 pages
dc26s006.pdf: 18 pages
dc26s009.pdf: 33 pages
dc26s010.pdf: 30 pages


In [62]:
# Extract text from the ADA guideline PDF documents

documents = []

for pdf in pdf_files:
    pdf_path = os.path.join(data_path, pdf)
    reader = PdfReader(pdf_path)

    text = ""

    for page in reader.pages:
        page_text = page.extract_text()

        if page_text:
            text += page_text + "\n"

    documents.append(
        {
            "filename": pdf,
            "text": text
        }
    )

print(f"Loaded {len(documents)} documents.")

Loaded 6 documents.


In [63]:
# Display basic information about the extracted documents

for document in documents:
    print("=" * 60)
    print(document["filename"])
    print(f"Characters: {len(document['text'])}")

dc26s002.pdf
Characters: 165958
dc26s004.pdf
Characters: 195695
dc26s005.pdf
Characters: 338666
dc26s006.pdf
Characters: 128520
dc26s009.pdf
Characters: 223271
dc26s010.pdf
Characters: 222366


In [64]:
# Preview the extracted text from the first ADA guideline document

print(documents[0]["text"][:2000])

2. DIAGNOSIS AND CLASSIFICATION OF DIABETES
2. Diagnosis and Classification of
Diabetes: Standards of Care in
Diabetes—2026
Diabetes Care 2026;49(Suppl. 1):S27–S49 | https://doi.org/10.2337/dc26-S002
American Diabetes Association 
Professional Practice 
Committee for 
Diabetes*
*A complete list of members of the American 
Diabetes Association 
Professional Practice
Committee for Diabetes can be found at https:// 
doi.org/10.2337/dc26-SINT.
Duality of interes t information for each contributor 
is available at https://doi.org/10.2337/dc26-SDIS.
Suggested citation: American Diabetes Association 
Professional 
Practice Committee for Diabetes. 
2. Diagnosis and classification of diabetes: 
Standards of Care in Diabetes—2026. Diabetes Care 
2026;49(Suppl. 1):S27–S49
© 2025 by the American Diabetes Association. 
Readers may 
use this work for educational, 
noncommercial purposes if properly cited and 
unaltered. The version of record may be linked 
at https://diabetesjournals.org/care, but A

In [65]:
# Clean extracted ADA guideline text before chunking

import re

def clean_guideline_text(text):
    # Join broken line breaks into spaces
    text = text.replace("\n", " ")

    # Remove downloaded-from text
    text = re.sub(r"Downloaded from .*?(?=S\d+|$)", " ", text)

    # Remove journal/header/footer phrases
    text = re.sub(r"Diabetes Care Volume 49, Supplement 1, January 2026", " ", text)
    text = re.sub(r"diabetesjournals\.org/care", " ", text)

    # Remove URLs
    text = re.sub(r"https?://\S+", " ", text)

    # Remove excessive whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()

cleaned_documents = []

for document in documents:
    cleaned_text = clean_guideline_text(document["text"])
    cleaned_documents.append(
        {
            "filename": document["filename"],
            "text": cleaned_text
        }
    )

print(f"Cleaned {len(cleaned_documents)} documents.")

Cleaned 6 documents.


In [66]:
# Preview the cleaned text from the first document

print(cleaned_documents[0]["text"][:3000])

2. DIAGNOSIS AND CLASSIFICATION OF DIABETES 2. Diagnosis and Classification of Diabetes: Standards of Care in Diabetes—2026 Diabetes Care 2026;49(Suppl. 1):S27–S49 | American Diabetes Association Professional Practice Committee for Diabetes* *A complete list of members of the American Diabetes Association Professional Practice Committee for Diabetes can be found at https:// doi.org/10.2337/dc26-SINT. Duality of interes t information for each contributor is available at Suggested citation: American Diabetes Association Professional Practice Committee for Diabetes. 2. Diagnosis and classification of diabetes: Standards of Care in Diabetes—2026. Diabetes Care 2026;49(Suppl. 1):S27–S49 © 2025 by the American Diabetes Association. Readers may use this work for educational, noncommercial purposes if properly cited and unaltered. The version of record may be linked at https:// , but ADA permission is required to post this work on any third-party site or platform. This publication and its cont

In [67]:
# Split cleaned ADA guideline text into smaller chunks for retrieval

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = []

for document in cleaned_documents:
    document_chunks = text_splitter.create_documents(
        [document["text"]],
        metadatas=[{"source": document["filename"]}]
    )
    chunks.extend(document_chunks)

print(f"Total cleaned chunks created: {len(chunks)}")

Total cleaned chunks created: 1531


In [68]:
# Preview a representative text chunk

print("Source:", chunks[10].metadata["source"])
print("\nChunk preview:\n")
print(chunks[10].page_content)

Source: dc26s002.pdf

Chunk preview:

. In this con- text, testing A1C helps determine the chronicity of hyperglycemia. However, in an individual without symptoms, FPG or 2-h PG can be used for screening and di - agnosis of diabetes. In nonpregnant indi- viduals, FPG (or A1C) is typically preferred for routine screening due to the ease of administration ( Table 2.3); however, the 2-h PG (OGTT) testing protocol is signifi- cantly more sensitive than the other two tests and is preferentially recommended for screening for some conditions (e.g., cystic fibrosis –related diabetes or post - transplantation diabetes mellitus). In the absence of classic symptoms of hypergly- cemia, repeat testing is required to con- firm the diagnosis regardless of the test used (see CONFIRMING THE DIAGNOSIS, below). Major advantages of glucose monitor- ing are its low cost and availability. Dis - advantages include the high diurnal variation in glucose and the 8-h fasting requirements ( Table 2.3)


In [69]:
# Load the embedding model

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully.


In [70]:
# Generate embeddings for all text chunks

chunk_texts = [chunk.page_content for chunk in chunks]

embeddings = embedding_model.encode(
    chunk_texts,
    show_progress_bar=True
)

print(f"Generated embeddings for {len(embeddings)} chunks.")
print(f"Embedding dimension: {len(embeddings[0])}")

Batches:   0%|          | 0/48 [00:00<?, ?it/s]

Generated embeddings for 1531 chunks.
Embedding dimension: 384


In [71]:
# Create a ChromaDB client and collection

client = chromadb.Client()

collection = client.get_or_create_collection(
    name="ada_diabetes_guidelines_cleaned"
)

print("ChromaDB collection created successfully.")

ChromaDB collection created successfully.


In [72]:
# Store text chunks, embeddings, and metadata in ChromaDB

ids = [f"chunk_{i}" for i in range(len(chunks))]
documents_text = [chunk.page_content for chunk in chunks]
metadatas = [chunk.metadata for chunk in chunks]

collection.add(
    ids=ids,
    documents=documents_text,
    embeddings=embeddings.tolist(),
    metadatas=metadatas
)

print(f"Stored {collection.count()} chunks in ChromaDB.")

Stored 1531 chunks in ChromaDB.


In [73]:
# Test basic retrieval using a sample Type 2 Diabetes management question

query = "What is the recommended HbA1c target for most adults with Type 2 Diabetes?"

query_embedding = embedding_model.encode([query])[0]

results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=3
)

print("Question:", query)
print("\nTop retrieved results:\n")

for i, document in enumerate(results["documents"][0]):
    print("=" * 80)
    print(f"Result {i + 1}")
    print("Source:", results["metadatas"][0][i]["source"])
    print(document[:1000])

Question: What is the recommended HbA1c target for most adults with Type 2 Diabetes?

Top retrieved results:

Result 1
Source: dc26s006.pdf
. Status of hemoglobin A1c measurement and goals for improvement: from chaos to order for improving diabetes care. Clin Chem 2011;57:205–214 5. Sacks DB, Kirkman MS, Little RR. Point-of-care HbA1c in clinical practice: caveats and considerations for optimal use. Diabetes Care 2024;47:1104–1110 6. Kidney Disease: Improving Global Outcomes (KDIGO) Work Group. KDIGO 2022 clinical practice guideline for diabetes management in chronic kidney disease. Kidney Int 2022;102:S1–S127 7. National Glycohemoglobin Standardization Program. HbA1c assay interferences. HbA1c methods: effects of hemoglobin variants (HbC, HbS, HbE and HbD traits) and elevated fetal hemoglobin (HbF). 2022. Accessed 14 August 2025. Available from 8. Sacks DB, Arnold M, Bakris GL, et al. Guidelines and recommendations for laboratory analysis in the diagnosis and management of diabetes me

In [74]:
# Create a reusable function for retrieving relevant guideline chunks

def retrieve_guideline_chunks(query, n_results=3):
    query_embedding = embedding_model.encode([query])[0]

    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=n_results
    )

    print("Question:", query)
    print("\nTop retrieved guideline chunks:\n")

    for i, document in enumerate(results["documents"][0]):
        print("=" * 80)
        print(f"Result {i + 1}")
        print("Source:", results["metadatas"][0][i]["source"])
        print(document[:1000])
        print()

In [75]:
# Test retrieval using multiple Type 2 Diabetes management questions

sample_questions = [
    "What is the recommended HbA1c target for most adults with Type 2 Diabetes?",
    "What medications are recommended for Type 2 Diabetes treatment?",
    "How should cardiovascular risk be managed in people with diabetes?",
    "What lifestyle changes are recommended for Type 2 Diabetes?",
    "How is diabetes diagnosed?"
]

for question in sample_questions:
    retrieve_guideline_chunks(question, n_results=3)
    print("\n\n")

Question: What is the recommended HbA1c target for most adults with Type 2 Diabetes?

Top retrieved guideline chunks:

Result 1
Source: dc26s006.pdf
. Status of hemoglobin A1c measurement and goals for improvement: from chaos to order for improving diabetes care. Clin Chem 2011;57:205–214 5. Sacks DB, Kirkman MS, Little RR. Point-of-care HbA1c in clinical practice: caveats and considerations for optimal use. Diabetes Care 2024;47:1104–1110 6. Kidney Disease: Improving Global Outcomes (KDIGO) Work Group. KDIGO 2022 clinical practice guideline for diabetes management in chronic kidney disease. Kidney Int 2022;102:S1–S127 7. National Glycohemoglobin Standardization Program. HbA1c assay interferences. HbA1c methods: effects of hemoglobin variants (HbC, HbS, HbE and HbD traits) and elevated fetal hemoglobin (HbF). 2022. Accessed 14 August 2025. Available from 8. Sacks DB, Arnold M, Bakris GL, et al. Guidelines and recommendations for laboratory analysis in the diagnosis and management of di